In [3]:
# data_agg_rnn_only.py
# Build ONLY an RNN-ready .npz (plus meta stored inside the npz)
# - X_seq: (N, SEQ_LEN_T, 6)  [per-game event counts]
# - X_static: (N, n_static)   [height, age, big5 + one-hot foot/pos]
# - y: (N,)                   [log1p(market_value_in_eur) by default]
# - player_id: (N,)
# - valuation_date: (N,)      [datetime64[ns]]

import numpy as np
import pandas as pd
from pathlib import Path

# ----------------------------
# Paths (portable, notebook-safe)
# ----------------------------
try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd().parent  # if you run it from a notebook folder

DATA_DIR = PROJECT_ROOT / "Data"
OUT_DIR = PROJECT_ROOT / "Data_Processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PLAYERS_CSV = DATA_DIR / "players.csv"
VALUATIONS_CSV = DATA_DIR / "player_valuations.csv"
EVENTS_CSV = DATA_DIR / "game_events.csv"

OUT_NPZ = OUT_DIR / "cumlag_rnn_dataset_only.npz"

# ----------------------------
# Config
# ----------------------------
SEQ_LEN_T = 20
MIN_PRIOR_GAMES = 3
USE_LOG_TARGET = True

# ----------------------------
# Helpers
# ----------------------------
def safe_to_datetime(s):
    return pd.to_datetime(s, errors="coerce", utc=False)

def compute_age_years(dob, ref_date):
    if pd.isna(dob) or pd.isna(ref_date):
        return np.nan
    return (ref_date - dob).days / 365.25

def standardize_position(pos):
    if pd.isna(pos):
        return "UNK"
    p = str(pos).upper()
    if "GOAL" in p or p == "GK":
        return "GK"
    if "DEF" in p:
        return "DEF"
    if "MID" in p:
        return "MID"
    if "ATT" in p or "FORW" in p or "WING" in p or "STRIK" in p:
        return "ATT"
    return p[:10]

def standardize_foot(foot):
    if pd.isna(foot):
        return "UNK"
    f = str(foot).lower()
    if f.startswith("right"):
        return "R"
    if f.startswith("left"):
        return "L"
    if "both" in f:
        return "B"
    return "UNK"

def make_big5_flag(val_df):
    BIG5_IDS = {"GB1", "ES1", "IT1", "DE1", "FR1"}
    comp = val_df["player_club_domestic_competition_id"].fillna("").astype(str).str.upper()
    val_df["is_big5_league"] = comp.isin(BIG5_IDS).astype(np.float32)
    return val_df

def to_int_player_id(df, col="player_id"):
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=[col])
    df[col] = df[col].astype(np.int64)
    return df

# ----------------------------
# Load data
# ----------------------------
print("Loading CSVs...")
players = pd.read_csv(PLAYERS_CSV, low_memory=False)
valuations = pd.read_csv(VALUATIONS_CSV, low_memory=False)
events = pd.read_csv(EVENTS_CSV, low_memory=False)

players = to_int_player_id(players, "player_id")
valuations = to_int_player_id(valuations, "player_id")

players["date_of_birth"] = safe_to_datetime(players.get("date_of_birth"))
valuations["date"] = safe_to_datetime(valuations.get("date"))

events["date"] = safe_to_datetime(events.get("date"))
events["game_id"] = pd.to_numeric(events.get("game_id"), errors="coerce")
events["minute"] = pd.to_numeric(events.get("minute"), errors="coerce")

# Ensure player id columns exist and are numeric where relevant
for c in ["player_id", "player_assist_id", "player_in_id"]:
    if c in events.columns:
        events[c] = pd.to_numeric(events[c], errors="coerce")

# Clean valuations
valuations = valuations.dropna(subset=["date", "market_value_in_eur"])
valuations["market_value_in_eur"] = pd.to_numeric(valuations["market_value_in_eur"], errors="coerce")
valuations = valuations.dropna(subset=["market_value_in_eur"])
valuations = valuations.sort_values(["player_id", "date"]).reset_index(drop=True)

# ----------------------------
# Static player features
# ----------------------------
print("Building static features...")

players_static = players[["player_id", "height_in_cm", "foot", "position", "date_of_birth"]].copy()
players_static["height_in_cm"] = pd.to_numeric(players_static["height_in_cm"], errors="coerce")

players_static["foot"] = players_static["foot"].apply(standardize_foot)
players_static["pos_group"] = players_static["position"].apply(standardize_position)

static_ohe = pd.get_dummies(
    players_static[["foot", "pos_group"]].fillna("UNK"),
    prefix=["foot", "pos"],
)

players_static_num = pd.concat(
    [
        players_static[["player_id", "height_in_cm", "date_of_birth"]].reset_index(drop=True),
        static_ohe.reset_index(drop=True),
    ],
    axis=1,
).drop_duplicates("player_id")

# ----------------------------
# Event-based per-game features
# ----------------------------
print("Building per-game event features...")

ev = events.dropna(subset=["date", "game_id"]).copy()
ev = ev.dropna(subset=["game_id"])
ev["game_id"] = ev["game_id"].astype(np.int64)

desc = ev.get("description", pd.Series([""] * len(ev))).fillna("")
etype = ev.get("type", pd.Series([""] * len(ev)))

is_goal = etype == "Goals"
is_yellow = (etype == "Cards") & desc.str.contains("Yellow card", case=False, na=False)
is_red = (etype == "Cards") & desc.str.contains("Red card", case=False, na=False)
is_sub = etype == "Substitutions"

def count_events(df, col, name):
    if col not in df.columns:
        return pd.DataFrame(columns=["player_id", "game_id", name])
    tmp = df[[col, "game_id"]].dropna().copy()
    tmp[col] = pd.to_numeric(tmp[col], errors="coerce")
    tmp = tmp.dropna(subset=[col])
    tmp[col] = tmp[col].astype(np.int64)
    out = (
        tmp.groupby([col, "game_id"])
        .size()
        .rename(name)
        .reset_index()
        .rename(columns={col: "player_id"})
    )
    return out

goals = count_events(ev[is_goal], "player_id", "goals")
assists = count_events(ev[is_goal], "player_assist_id", "assists")
yellow = count_events(ev[is_yellow], "player_id", "yellow_cards")
red = count_events(ev[is_red], "player_id", "red_cards")
sub_in = count_events(ev[is_sub], "player_in_id", "sub_in")
sub_out = count_events(ev[is_sub], "player_id", "sub_out")

game_dates = ev.groupby("game_id")["date"].min().reset_index(name="game_date")

pairs = pd.concat([goals, assists, yellow, red, sub_in, sub_out], axis=0)[["player_id", "game_id"]]
pairs = pairs.dropna().drop_duplicates()

per_game = pairs.merge(game_dates, on="game_id", how="left")

for df in [goals, assists, yellow, red, sub_in, sub_out]:
    per_game = per_game.merge(df, on=["player_id", "game_id"], how="left")

per_game = per_game.fillna(0.0)
per_game = per_game.dropna(subset=["game_date"])
per_game = per_game.sort_values(["player_id", "game_date"]).reset_index(drop=True)

GAME_FEATURES = ["goals", "assists", "yellow_cards", "red_cards", "sub_in", "sub_out"]

# Group per player (kept simple; can be memory heavy but OK for your dataset size)
pgroups = {pid: g for pid, g in per_game.groupby("player_id")}
# Merge valuations with static info
val = valuations.merge(players_static_num, on="player_id", how="left")
val = make_big5_flag(val)

# Compute age per valuation
val["age_years"] = val.apply(lambda r: compute_age_years(r["date_of_birth"], r["date"]), axis=1)

val["y_raw"] = pd.to_numeric(val["market_value_in_eur"], errors="coerce").astype(np.float32)
val["y_log"] = np.log1p(val["y_raw"]).astype(np.float32)

static_cols = ["height_in_cm", "age_years", "is_big5_league"] + [
    c for c in val.columns if c.startswith("foot_") or c.startswith("pos_")
]

# IMPORTANT: avoid NaNs in X_static (you can also choose to drop rows later)
val[static_cols] = val[static_cols].fillna(0.0)

vgroups = {pid: g.sort_values("date").reset_index(drop=True) for pid, g in val.groupby("player_id")}

# ----------------------------
# Build ONLY RNN arrays
# ----------------------------
print("Building RNN arrays...")

X_seq_list = []
X_static_list = []
y_list = []
pid_list = []
date_list = []

for pid, vg in vgroups.items():
    if pid not in pgroups:
        continue

    gg = pgroups[pid]
    g_dates = gg["game_date"].to_numpy()
    g_feats = gg[GAME_FEATURES].to_numpy(dtype=np.float32)

    val_dates = vg["date"].to_numpy()
    idxs = np.searchsorted(g_dates, val_dates, side="left")

    for i, n_before in enumerate(idxs):
        if n_before < MIN_PRIOR_GAMES:
            continue

        seq = g_feats[max(0, n_before - SEQ_LEN_T):n_before]
        if seq.shape[0] < SEQ_LEN_T:
            pad = np.zeros((SEQ_LEN_T - seq.shape[0], seq.shape[1]), dtype=np.float32)
            seq = np.vstack([pad, seq])

        y_raw = float(vg.loc[i, "y_raw"])
        y_log = float(vg.loc[i, "y_log"])

        X_seq_list.append(seq)
        X_static_list.append(vg.loc[i, static_cols].to_numpy(dtype=np.float32))
        y_list.append(y_log if USE_LOG_TARGET else y_raw)
        pid_list.append(pid)
        date_list.append(vg.loc[i, "date"])

X_seq = np.asarray(X_seq_list, dtype=np.float32)
X_static = np.asarray(X_static_list, dtype=np.float32)
y_out = np.asarray(y_list, dtype=np.float32)

player_id_arr = np.asarray(pid_list, dtype=np.int64)
valuation_date_arr = np.asarray(pd.to_datetime(date_list).to_numpy(), dtype="datetime64[ns]")

print("X_seq:", X_seq.shape, "X_static:", X_static.shape, "y:", y_out.shape)
print("Static dims:", X_static.shape[1])

# ----------------------------
# Save ONE npz (with meta inside)
# ----------------------------
np.savez_compressed(
    OUT_NPZ,
    X_seq=X_seq,
    X_static=X_static,
    y=y_out,
    player_id=player_id_arr,
    valuation_date=valuation_date_arr,
)

print("Saved:", OUT_NPZ)


Loading CSVs...
Building static features...
Building per-game event features...
Building RNN arrays...
X_seq: (278558, 20, 6) X_static: (278558, 12) y: (278558,)
Static dims: 12
Saved: c:\Users\johan\OneDrive\IN2010\FYSSTK3155\PROJECT 3\Code\Data_Processed\cumlag_rnn_dataset_only.npz
